# Day 33 — Dynamic programming from the ground up

Dynamic programming is not an algorithm, it is a rewrite. A recursion whose call
tree keeps asking the same questions gets folded into a table that is filled once,
left to right.

Two problems carry the whole idea:

* **climbing stairs** — count the ways, combine with `+`
* **coin change** — find the cheapest way, combine with `min`

and a third one you already met on Day 08, **word break**, turns out to be the same
table with `or`. The interesting part is not the recurrence; it is the *order* the
table is filled in, because getting that order wrong produces a program that runs
happily and answers a different question.

## 1. What the naive recursion actually repeats

`f(n) = f(n-1) + f(n-2)` is correct. It is also exponential, because nothing
remembers an answer that was already computed.

In [1]:
CALLS = {'n': 0}

def stairs_naive(n):
    """Ways to climb n steps taking 1 or 2 at a time. Exponential."""
    CALLS['n'] += 1
    if n < 0:
        return 0
    if n == 0:
        return 1                       # one way to stand still
    return stairs_naive(n - 1) + stairs_naive(n - 2)

for n in (10, 20, 25, 30):
    CALLS['n'] = 0
    val = stairs_naive(n)
    print(f'n = {n:>2}  ways {val:>7,}  calls {CALLS["n"]:>9,}'
          f'  distinct subproblems {n + 1:>3}'
          f'  waste {CALLS["n"] / (n + 1):>9,.0f}x')

n = 10  ways      89  calls       287  distinct subproblems  11  waste        26x
n = 20  ways  10,946  calls    35,421  distinct subproblems  21  waste     1,687x
n = 25  ways 121,393  calls   392,835  distinct subproblems  26  waste    15,109x
n = 30  ways 1,346,269  calls 4,356,617  distinct subproblems  31  waste   140,536x


The tree is as big as the answer it returns — `stairs_naive(30)` makes 4,356,617
calls to compute 1,346,269, and there are only 31 different questions in there.

## 2. Three shapes of one table

Memoisation keeps the recursion and adds a dictionary. Tabulation drops the
recursion and fills the array. Both compute each cell once; the rolling version
notices that only the last two cells are ever read.

In [2]:
def stairs_memo(n):
    memo = {}
    def go(k):
        if k < 0:
            return 0
        if k == 0:
            return 1
        if k not in memo:
            memo[k] = go(k - 1) + go(k - 2)
        return memo[k]
    return go(n), memo

def stairs_table(n):
    dp = [0] * (n + 1)
    dp[0] = 1                          # the empty climb
    for i in range(1, n + 1):
        dp[i] = dp[i - 1]              # arrive with a 1-step
        if i >= 2:
            dp[i] += dp[i - 2]         # arrive with a 2-step
    return dp

def stairs_rolling(n):
    prev, cur = 1, 1
    for _ in range(2, n + 1):
        prev, cur = cur, cur + prev
    return cur if n >= 1 else 1

val, memo = stairs_memo(10)
dp = stairs_table(10)
print('memoised  ', val, 'with', len(memo), 'memo entries')
print('tabulated ', dp)
print('rolling   ', stairs_rolling(10), '(two variables, O(1) space)')
assert val == dp[10] == stairs_rolling(10) == 89

memoised   89 with 10 memo entries
tabulated  [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]
rolling    89 (two variables, O(1) space)


`dp[i] = dp[i-1] + dp[i-2]` is Fibonacci. That is the giveaway: *counting* problems
add, they do not minimise.

## 3. Coin change — fewest coins (LeetCode 322)

Same table, different combine step. `dp[a]` is the best answer for amount `a`, and
an unreachable amount stays at infinity.

In [3]:
INF = float('inf')

def coin_change_min_trace(coins, amount):
    """Fewest coins for amount, plus the coin that produced each cell."""
    dp = [0] + [INF] * amount
    pick = [None] * (amount + 1)
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:
                dp[a] = dp[a - c] + 1
                pick[a] = c            # remember how we got here
    if dp[amount] == INF:
        return -1, [], dp
    out, a = [], amount
    while a > 0:                       # walk the parent pointers back
        out.append(pick[a])
        a -= pick[a]
    return dp[amount], sorted(out, reverse=True), dp

def coin_change_min_greedy(coins, amount):
    left, used = amount, []
    for c in sorted(coins, reverse=True):
        while c <= left:
            left -= c
            used.append(c)
    return (-1, []) if left else (len(used), used)

coins, amount = [1, 5, 6, 9], 11
best, used, dp = coin_change_min_trace(coins, amount)
print('dp   ', [('-' if v == INF else v) for v in dp])
print('best ', best, 'coins:', used)
print('greedy', coin_change_min_greedy(coins, amount))
print('unreachable:', coin_change_min_trace([5, 7], 3)[0])
assert best == 2 and used == [6, 5]

dp    [0, 1, 2, 3, 4, 1, 1, 2, 3, 1, 2, 2]
best  2 coins: [6, 5]
greedy (3, [9, 1, 1])
unreachable: -1


Greedy grabs the 9 and is then stuck with `1 + 1`; the table takes `5 + 6`. Greedy
is safe on a *designed* currency, not on an arbitrary coin set — and the `pick`
array is what turns a number into an actual answer you can hand back.

## 4. The silent bug: combinations vs permutations

These two functions differ by which loop is outer. Nothing raises. One of them
answers a different question.

In [4]:
def coin_ways_combinations(coins, amount):
    """LC 518. Coin loop OUTSIDE: each coin is offered once, so order is fixed."""
    dp = [1] + [0] * amount
    for c in coins:                    # <- outer
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]

def coin_ways_permutations(coins, amount):
    """LC 377. Amount loop OUTSIDE: every coin may be the *last* one added."""
    dp = [1] + [0] * amount
    for a in range(1, amount + 1):     # <- outer
        for c in coins:
            if c <= a:
                dp[a] += dp[a - c]
    return dp[amount]

print('combinations', coin_ways_combinations([1, 2, 5], 5))
print('permutations', coin_ways_permutations([1, 2, 5], 5))

combinations 4
permutations 9


Brute force, to prove which is which.

In [5]:
def enumerate_combinations(coins, amount):
    out = []
    def go(i, left, cur):
        if left == 0:
            out.append(tuple(cur)); return
        if i == len(coins):
            return
        go(i + 1, left, cur)                     # skip this coin for good
        if coins[i] <= left:
            cur.append(coins[i])
            go(i, left - coins[i], cur)          # reuse is allowed
            cur.pop()
    go(0, amount, [])
    return out

def enumerate_permutations(coins, amount):
    out = []
    def go(left, cur):
        if left == 0:
            out.append(tuple(cur)); return
        for c in coins:
            if c <= left:
                cur.append(c)
                go(left - c, cur)
                cur.pop()
    go(amount, [])
    return out

cs = enumerate_combinations([1, 2, 5], 5)
ps = enumerate_permutations([1, 2, 5], 5)
print('multisets :', [' + '.join(map(str, c)) for c in sorted(cs)])
print('sequences :', len(ps), 'e.g.', ps[:3])
assert coin_ways_combinations([1, 2, 5], 5) == len(cs) == 4
assert coin_ways_permutations([1, 2, 5], 5) == len(ps) == 9

print()
print(' amount  combinations  permutations')
for a in (5, 10, 20, 30):
    print(f' {a:>6}  {coin_ways_combinations([1, 2, 5], a):>12,}'
          f'  {coin_ways_permutations([1, 2, 5], a):>13,}')

multisets : ['1 + 1 + 1 + 1 + 1', '1 + 1 + 1 + 2', '1 + 2 + 2', '5']
sequences : 9 e.g. [(1, 1, 1, 1, 1), (1, 1, 1, 2), (1, 1, 2, 1)]

 amount  combinations  permutations
      5             4              9
     10            10            128
     20            29         26,547
     30            58      5,508,222


Read the loops out loud and the difference stops being a rule to memorise:

* coin loop outside — coin 5 is offered once and for all, so `1+2+2` is built in
  exactly one order;
* amount loop outside — every coin gets a turn at being the last one added, so
  `1+2+2`, `2+1+2` and `2+2+1` are three different fills.

## 5. Climbing stairs *is* coin change with coins {1, 2}

Steps are ordered: 1-then-2 is a different climb from 2-then-1. So stairs must match
the **permutation** table, not the combination one.

In [6]:
for n in range(1, 11):
    a = stairs_rolling(n)
    b = coin_ways_permutations([1, 2], n)
    c = coin_ways_combinations([1, 2], n)
    print(f'n = {n:>2}   stairs {a:>3}   permutations {b:>3}   combinations {c:>3}')
    assert a == b
print()
print('the combination column counts how many 2-steps you took, not how you climbed')

n =  1   stairs   1   permutations   1   combinations   1
n =  2   stairs   2   permutations   2   combinations   2
n =  3   stairs   3   permutations   3   combinations   2
n =  4   stairs   5   permutations   5   combinations   3
n =  5   stairs   8   permutations   8   combinations   3
n =  6   stairs  13   permutations  13   combinations   4
n =  7   stairs  21   permutations  21   combinations   4
n =  8   stairs  34   permutations  34   combinations   5
n =  9   stairs  55   permutations  55   combinations   5
n = 10   stairs  89   permutations  89   combinations   6

the combination column counts how many 2-steps you took, not how you climbed


## 6. Day 08 word break, re-read

Same loop, same states. The combine step is `or` — and swapping it for `+` counts
the segmentations instead of deciding whether one exists.

In [7]:
def word_break(s, words):
    """dp[i] = can s[:i] be segmented?"""
    wordset = set(words)
    dp = [True] + [False] * len(s)
    for i in range(1, len(s) + 1):
        for j in range(i):
            if dp[j] and s[j:i] in wordset:
                dp[i] = True
                break
    return dp[len(s)]

def word_break_count(s, words):
    """Swap `or` for `+` and the same table counts the segmentations."""
    wordset = set(words)
    dp = [1] + [0] * len(s)
    for i in range(1, len(s) + 1):
        for j in range(i):
            if dp[j] and s[j:i] in wordset:
                dp[i] += dp[j]
    return dp[len(s)]

print(word_break('applepenapple', ['apple', 'pen']),
      word_break_count('applepenapple', ['apple', 'pen']))
print(word_break('catsandog', ['cats', 'dog', 'sand', 'and', 'cat']))
print()
print('stairs       dp[i] = sum over steps        combine with +   -> count')
print('coin change  dp[a] = 1 + min over coins    combine with min -> optimise')
print('word break   dp[i] = or over cut points    combine with or  -> decide')
assert word_break('applepenapple', ['apple', 'pen']) is True
assert word_break_count('applepenapple', ['apple', 'pen']) == 1

True 1
False

stairs       dp[i] = sum over steps        combine with +   -> count
coin change  dp[a] = 1 + min over coins    combine with min -> optimise
word break   dp[i] = or over cut points    combine with or  -> decide


## 7. Memoisation vs tabulation — who touches fewer states

Top-down only computes states that are actually asked for. When the reachable
states are sparse, that is a real saving; when every state is reachable, tabulation
wins instead, because it has no call overhead and no call stack to blow.

In [8]:
import sys
from functools import lru_cache

def coin_change_min_memo(coins, amount):
    visited = set()
    @lru_cache(maxsize=None)
    def best(a):
        visited.add(a)
        if a == 0:
            return 0
        if a < 0:
            return INF
        return min((best(a - c) + 1 for c in coins), default=INF)
    sys.setrecursionlimit(100000)
    r = best(amount)
    best.cache_clear()
    return (-1 if r == INF else r), len(visited)

for coins, amount in (([100, 250], 10000), ([1, 2, 5], 1000)):
    r, states = coin_change_min_memo(coins, amount)
    print(f'coins {str(coins):<10} amount {amount:>6}  answer {r:>4}'
          f'  memo states {states:>5}  table cells {amount + 1:>6}'
          f'  ratio {(amount + 1) / states:>5.1f}x')
print()
print('without setrecursionlimit, the {1,2,5} case dies: the chain')
print('1000 -> 999 -> 998 -> ... is exactly as deep as the default limit')

coins [100, 250] amount  10000  answer   40  memo states   203  table cells  10001  ratio  49.3x
coins [1, 2, 5]  amount   1000  answer  200  memo states  1005  table cells   1001  ratio   1.0x

without setrecursionlimit, the {1,2,5} case dies: the chain
1000 -> 999 -> 998 -> ... is exactly as deep as the default limit


## 8. LeetCode round-up

70 climbing stairs, 322 coin change, 518 coin change II, 377 combination sum IV,
139 word break. Note that LC 377 is *named* after combinations and asks for
permutations — the examples are right, the title is not.

In [9]:
def lc70_climb_stairs(n):
    a, b = 1, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

print('70  climbStairs(10)              =', lc70_climb_stairs(10))
print('322 coinChange([1,2,5], 11)      =', coin_change_min_trace([1, 2, 5], 11)[0])
print('322 coinChange([2], 3)           =', coin_change_min_trace([2], 3)[0])
print('518 change(5, [1,2,5])           =', coin_ways_combinations([1, 2, 5], 5))
print('377 combinationSum4([1,2,3], 4)  =', coin_ways_permutations([1, 2, 3], 4))
print('139 wordBreak("leetcode", ...)   =', word_break('leetcode', ['leet', 'code']))

assert lc70_climb_stairs(10) == 89
assert coin_change_min_trace([1, 2, 5], 11)[0] == 3
assert coin_change_min_trace([2], 3)[0] == -1
assert coin_ways_combinations([1, 2, 5], 5) == 4
assert coin_ways_permutations([1, 2, 3], 4) == 7
assert word_break('leetcode', ['leet', 'code']) is True
print()
print('all assertions passed')

70  climbStairs(10)              = 89
322 coinChange([1,2,5], 11)      = 3
322 coinChange([2], 3)           = -1
518 change(5, [1,2,5])           = 4
377 combinationSum4([1,2,3], 4)  = 7
139 wordBreak("leetcode", ...)   = True

all assertions passed


## Takeaways

* DP = a recursion with overlapping subproblems, folded into a table filled once.
* The recurrence is the easy half; the **fill order** is the half that bites.
* Coin loop outside counts combinations, amount loop outside counts permutations —
  same shape, different question, no error message.
* Keep a `pick`/parent array if you need the answer and not just its cost.
* Same table, three operators: `+` counts, `min` optimises, `or` decides.